## Clustering de Clientes con K-Means

### Por: Grupo 12 - ITBA
### Fecha: 2026-04-15 (actualizado 2026-06-22)

### Descripcion:
Segmentacion de 4,338 clientes usando K-Means sobre features RFM + atributos de producto.
Se determina k optimo, se entrenan los clusters y se interpretan los segmentos resultantes.

**Mejora post-feedback (Entrega 03):** la devolucion senalo un silhouette bajo (~0.17),
indicando clusters poco separados. Se revisaron las features y el metodo:
1. Se **excluye `Cancel_rate`** del clustering (64% de ceros y outliers extremos: aporta
   ruido). Se conserva como descriptor de negocio y como feature del modelo de churn.
2. Se agrega **PCA** (reduccion de dimensionalidad, U4) antes de K-Means para
   de-correlacionar y compactar el espacio.

Resultado: el silhouette a k=4 sube de ~0.17 a ~0.31 (+80%), manteniendo los 4 segmentos
de negocio y el aporte de la afinidad de producto.

In [1]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import calinski_harabasz_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

FEATURE_PATH = Path("../../data/04_feature/")
MODEL_PATH = Path("../../data/06_models/")
OUTPUT_PATH = Path("../../data/07_model_output/")
REPORT_PATH = Path("../../data/08_reporting/")

for p in [MODEL_PATH, OUTPUT_PATH, REPORT_PATH]:
    p.mkdir(parents=True, exist_ok=True)

## Cargar datos

In [2]:
rfm = pd.read_parquet(FEATURE_PATH / "rfm_clientes_enriched.parquet")
print(f"Clientes: {len(rfm):,}")
print(f"Columnas: {list(rfm.columns)}")
rfm.head()

Clientes: 4,338
Columnas: ['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Cancel_rate', 'dominant_color', 'pct_with_color', 'color_diversity', 'is_color_specialist', 'dominant_material', 'pct_with_material', 'avg_quantity_in_set', 'pct_purchases_sets']


,CustomerID,Recency,Frequency,Monetary,Cancel_rate,dominant_color,pct_with_color,color_diversity,is_color_specialist,dominant_material,pct_with_material,avg_quantity_in_set,pct_purchases_sets
0,12346,326,1,0.00,100.0,red,0.000000,0.000000,True,ceramic,100.000000,0.000000,0.000000
1,12347,2,7,4310.00,0.0,red,34.065934,2.577407,False,wooden,3.846154,38.307692,7.142857
2,12348,75,4,1797.24,0.0,pink,29.032258,2.113283,False,paper,6.451613,35.785714,45.161290
3,12349,19,1,1757.55,0.0,red,27.397260,1.970951,False,tin,21.917808,6.888889,12.328767
4,12350,310,1,334.40,0.0,red,29.411765,1.521928,False,metal,29.411765,0.000000,0.000000


## Preparacion de features

Seleccionamos features numericos. Aplicamos log1p a variables con alta asimetria
(Frequency, Monetary, Cancel_rate) para reducir el efecto de outliers en K-Means.

In [3]:
# Features para clustering
# Mejora post-feedback (Entrega 03): se EXCLUYE Cancel_rate del clustering. Tiene 64%
# de ceros y outliers extremos (std altisima), por lo que aporta ruido y degrada la
# separacion de clusters. Se mantiene como descriptor de negocio y como feature del
# modelo de churn, pero no como input de la segmentacion.
FEATURES = [
    "Recency",
    "Frequency",
    "Monetary",
    "pct_with_color",
    "color_diversity",
    "pct_with_material",
    "avg_quantity_in_set",
    "pct_purchases_sets",
]

X_raw = rfm[FEATURES].copy()

# Preprocesamiento empaquetado en un Pipeline para que la inferencia reproduzca
# exactamente el entrenamiento:
#   1) log1p en variables muy asimetricas (Frequency, Monetary),
#   2) estandarizacion,
#   3) PCA: reduce dimensionalidad y de-correlaciona el espacio (tecnica de la U4),
#      lo que compacta los clusters y mejora el silhouette.
# El artefacto serializado es autocontenido: kmeans.predict() funciona con valores
# crudos sin replicar transformaciones por fuera.
LOG_FEATURES = ["Frequency", "Monetary"]
OTHER_FEATURES = [f for f in FEATURES if f not in LOG_FEATURES]

N_COMPONENTS = 3  # ~64% de varianza; mejor balance silhouette / interpretabilidad

preprocess = Pipeline(
    [
        (
            "log",
            ColumnTransformer(
                [
                    (
                        "log1p",
                        FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
                        LOG_FEATURES,
                    ),
                    ("passthrough", "passthrough", OTHER_FEATURES),
                ]
            ),
        ),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=N_COMPONENTS, random_state=42)),
    ]
)
X_scaled = preprocess.fit_transform(X_raw)

evr = preprocess.named_steps["pca"].explained_variance_ratio_
print(f"Shape tras PCA: {X_scaled.shape}")
print(f"Varianza explicada por componente: {np.round(evr, 3)}")
print(f"Varianza explicada acumulada: {np.round(np.cumsum(evr), 3)}")

Shape tras PCA: (4338, 3)
Varianza explicada por componente: [0.325 0.181 0.136]
Varianza explicada acumulada: [0.325 0.506 0.641]


## Seleccion de k optimo

In [4]:
K_RANGE = range(2, 11)
inertias = []
silhouettes = []
calinskis = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    calinskis.append(calinski_harabasz_score(X_scaled, labels))
    print(f"k={k}: silhouette={silhouettes[-1]:.4f}, calinski={calinskis[-1]:.1f}")

k=2: silhouette=0.3141, calinski=2185.5


k=3: silhouette=0.3177, calinski=1925.6


k=4: silhouette=0.3143, calinski=1903.2


k=5: silhouette=0.2715, calinski=1870.3


k=6: silhouette=0.2677, calinski=1742.4


k=7: silhouette=0.2450, calinski=1636.4


k=8: silhouette=0.2465, calinski=1561.9


k=9: silhouette=0.2464, calinski=1511.0


k=10: silhouette=0.2354, calinski=1458.8


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(K_RANGE, inertias, "bo-")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(K_RANGE, silhouettes, "ro-")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score vs k")

axes[2].plot(K_RANGE, calinskis, "go-")
axes[2].set_xlabel("k")
axes[2].set_ylabel("Calinski-Harabasz")
axes[2].set_title("Calinski-Harabasz vs k")

plt.tight_layout()
plt.savefig(REPORT_PATH / "clustering_k_selection.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_23425/1698051114.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Entrenar modelo final

In [6]:
# k se mantiene en 4 para preservar los 4 segmentos de negocio (VIP, Compradores de
# Sets, En Riesgo, Dormidos). Con el nuevo preprocesamiento (sin Cancel_rate + PCA) el
# silhouette a k=4 sube de ~0.17 a ~0.31, respondiendo al feedback de la Entrega 03.
best_k = 4

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, rfm["Cluster"])
cal = calinski_harabasz_score(X_scaled, rfm["Cluster"])

print(f"Modelo final: k={best_k}")
print(f"Silhouette score: {sil:.4f}")
print(f"Calinski-Harabasz: {cal:.1f}")
print("\nDistribucion de clusters:")
print(rfm["Cluster"].value_counts().sort_index())

Modelo final: k=4
Silhouette score: 0.3143
Calinski-Harabasz: 1903.2

Distribucion de clusters:
Cluster
0    1787
1     981
2     605
3     965
Name: count, dtype: int64


## Analisis de centroides

In [7]:
# Promedios por cluster (valores originales, no escalados)
centroid_cols = [*FEATURES, "Cluster"]
centroids = (
    rfm[
        [
            "Recency",
            "Frequency",
            "Monetary",
            "Cancel_rate",
            "pct_with_color",
            "color_diversity",
            "pct_with_material",
            "avg_quantity_in_set",
            "pct_purchases_sets",
            "Cluster",
        ]
    ]
    .groupby("Cluster")
    .mean()
    .round(2)
)

print("Centroides (valores originales promedio):")
print(centroids.to_string())

print("\nTamano de clusters:")
for c in sorted(rfm["Cluster"].unique()):
    n = (rfm["Cluster"] == c).sum()
    print(f"  Cluster {c}: {n} clientes ({n / len(rfm) * 100:.1f}%)")

Centroides (valores originales promedio):
         Recency  Frequency  Monetary  Cancel_rate  pct_with_color  color_diversity  pct_with_material  avg_quantity_in_set  pct_purchases_sets
Cluster                                                                                                                                        
0          32.69       7.94   3934.87         2.25           28.91             2.35              15.98                12.48                7.50
1         170.75       1.53    428.35         2.45           44.98             1.75              17.13                 3.90                3.67
2         112.63       2.03    708.84         2.57           23.77             1.61              13.99                29.81               21.03
3         111.25       1.67    448.31         9.57           13.82             0.68              21.05                 3.56                4.97

Tamano de clusters:
  Cluster 0: 1787 clientes (41.2%)
  Cluster 1: 981 clientes (22.6%)
  Cl

## Visualizacion: Boxplots por cluster

In [8]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    rfm.boxplot(column=feat, by="Cluster", ax=axes[i])
    axes[i].set_title(feat)
    axes[i].set_xlabel("Cluster")

plt.suptitle("Distribucion de Features por Cluster", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(REPORT_PATH / "clustering_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_23425/3593562816.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analisis de variables categoricas por cluster

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Dominant color por cluster
for c in sorted(rfm["Cluster"].unique()):
    subset = rfm[rfm["Cluster"] == c]
    top_colors = subset["dominant_color"].value_counts().head(5)
    print(f"Cluster {c} - Top 5 colores: {dict(top_colors)}")

# Dominant material por cluster
print()
for c in sorted(rfm["Cluster"].unique()):
    subset = rfm[rfm["Cluster"] == c]
    top_mats = subset["dominant_material"].value_counts().head(5)
    print(f"Cluster {c} - Top 5 materiales: {dict(top_mats)}")

# Grafico: color specialist por cluster
specialist_rate = rfm.groupby("Cluster")["is_color_specialist"].mean() * 100
specialist_rate.plot(kind="bar", ax=axes[0], color="steelblue", alpha=0.8)
axes[0].set_title("% Color Specialists por Cluster")
axes[0].set_ylabel("%")
axes[0].set_xlabel("Cluster")
axes[0].tick_params(axis="x", rotation=0)

# Grafico: cancel rate por cluster
cancel_rate = rfm.groupby("Cluster")["Cancel_rate"].mean()
cancel_rate.plot(kind="bar", ax=axes[1], color="salmon", alpha=0.8)
axes[1].set_title("Cancel Rate promedio por Cluster")
axes[1].set_ylabel("Cancel Rate (%)")
axes[1].set_xlabel("Cluster")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(REPORT_PATH / "clustering_categorical.png", dpi=150, bbox_inches="tight")
plt.show()

Cluster 0 - Top 5 colores: {'red': np.int64(989), 'pink': np.int64(283), 'white': np.int64(266), 'blue': np.int64(87), 'silver': np.int64(66)}
Cluster 1 - Top 5 colores: {'red': np.int64(386), 'pink': np.int64(213), 'white': np.int64(173), 'blue': np.int64(55), 'cream': np.int64(49)}
Cluster 2 - Top 5 colores: {'red': np.int64(330), 'pink': np.int64(169), 'white': np.int64(38), 'blue': np.int64(18), 'cream': np.int64(11)}
Cluster 3 - Top 5 colores: {'red': np.int64(503), 'white': np.int64(134), 'pink': np.int64(115), 'silver': np.int64(60), 'blue': np.int64(49)}

Cluster 0 - Top 5 materiales: {'metal': np.int64(424), 'paper': np.int64(311), 'tin': np.int64(309), 'wooden': np.int64(276), 'glass': np.int64(250)}
Cluster 1 - Top 5 materiales: {'tin': np.int64(284), 'metal': np.int64(209), 'glass': np.int64(139), 'paper': np.int64(114), 'wood': np.int64(87)}
Cluster 2 - Top 5 materiales: {'paper': np.int64(195), 'tin': np.int64(174), 'metal': np.int64(75), 'wooden': np.int64(65), 'glass': 

/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_23425/783574822.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Heatmap de centroides normalizados

In [10]:
# Normalizar centroides para heatmap (min-max por feature)
centroids_norm = centroids.copy()
for col in centroids_norm.columns:
    cmin = centroids_norm[col].min()
    cmax = centroids_norm[col].max()
    if cmax > cmin:
        centroids_norm[col] = (centroids_norm[col] - cmin) / (cmax - cmin)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    centroids_norm.T, annot=centroids.T.values, fmt=".1f", cmap="YlOrRd", ax=ax, linewidths=0.5
)
ax.set_title("Perfil de Segmentos (valores originales, color = intensidad relativa)", fontsize=14)
ax.set_xlabel("Cluster")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.savefig(REPORT_PATH / "clustering_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_23425/3342990884.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Etiquetas de negocio

Basandose en el perfil de centroides (valores en el espacio original de features):

| Cluster | Etiqueta | Perfil | % Clientes | % Revenue |
|---------|----------|--------|------------|-----------|
| 0 | **VIP** | **Recency baja (~33d), alta frecuencia (~7.9), alto monetary (~$3,935)**; concentra el grueso del revenue | 41.2% | 84.6% |
| 1 | **Dormidos** | **Recency alta (~171d)**, baja frecuencia (~1.5) y bajo monetary | 22.6% | 5.1% |
| 2 | **Compradores de Sets** | **alto avg_quantity_in_set (~29.8) y pct_purchases_sets (~21%)** | 13.9% | 5.2% |
| 3 | **En Riesgo** | **cancel rate promedio alto (~9.6%)** y baja diversidad de color | 22.2% | 5.2% |

Nota: `Cancel_rate` ya no es feature de clustering, pero sigue discriminando al segmento
**En Riesgo** (cancel rate ~9.6% vs ~2.3% en el resto), lo que valida la coherencia de los grupos.

In [11]:
# Etiquetas de negocio asignadas segun el perfil de cada cluster (ver celda anterior):
#   - VIP: mayor Monetary/Frequency y menor Recency (concentra el grueso del revenue).
#   - Compradores de Sets: mayor pct_purchases_sets y avg_quantity_in_set.
#   - En Riesgo: mayor Cancel_rate promedio (descriptor, no feature de clustering).
#   - Dormidos: alta Recency y baja Frequency (clientes inactivos).
# El mapeo de indices es estable dado random_state=42 y n_init=20; se valida con el
# perfil de centroides de la celda 12.
LABELS = {0: "VIP", 1: "Dormidos", 2: "Compradores de Sets", 3: "En Riesgo"}

rfm["Segment_label"] = rfm["Cluster"].map(LABELS)

# Resumen por segmento
summary = (
    rfm.groupby(["Cluster", "Segment_label"])
    .agg(
        n_clientes=("CustomerID", "count"),
        recency_mean=("Recency", "mean"),
        frequency_mean=("Frequency", "mean"),
        monetary_mean=("Monetary", "mean"),
        monetary_total=("Monetary", "sum"),
        cancel_rate_mean=("Cancel_rate", "mean"),
    )
    .round(1)
)

summary["pct_clientes"] = (summary["n_clientes"] / summary["n_clientes"].sum() * 100).round(1)
summary["pct_revenue"] = (summary["monetary_total"] / summary["monetary_total"].sum() * 100).round(
    1
)

print("Resumen de segmentos:")
print(summary.to_string())

Resumen de segmentos:
                             n_clientes  recency_mean  frequency_mean  monetary_mean  monetary_total  cancel_rate_mean  pct_clientes  pct_revenue
Cluster Segment_label                                                                                                                            
0       VIP                        1787          32.7             7.9         3934.9       7031614.2               2.2          41.2         84.6
1       Dormidos                    981         170.7             1.5          428.4        420213.8               2.4          22.6          5.1
2       Compradores de Sets         605         112.6             2.0          708.8        428846.0               2.6          13.9          5.2
3       En Riesgo                   965         111.3             1.7          448.3        432620.9               9.6          22.2          5.2


## Visualización 2D de los segmentos (t-SNE)

Proyectamos el espacio PCA a 2D con **t-SNE** (técnica de reducción no lineal, U4 del temario) para *ver* la separación de los segmentos. Complementa al silhouette: clusters bien separados aparecen como nubes distinguibles.

In [12]:
# t-SNE sobre el espacio PCA usado por K-Means (mismo X_scaled del pipeline).
tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=42)
emb = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(9, 7))
palette = {
    "VIP": "#2ecc71",
    "Compradores de Sets": "#3498db",
    "En Riesgo": "#e74c3c",
    "Dormidos": "#95a5a6",
}
for cl in sorted(rfm["Cluster"].unique()):
    mask = rfm["Cluster"] == cl
    label = LABELS.get(cl, f"Cluster {cl}")
    ax.scatter(emb[mask, 0], emb[mask, 1], s=8, alpha=0.5, label=label, color=palette.get(label))
ax.set_title(f"Proyección t-SNE de los segmentos (silhouette={sil:.3f})")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(markerscale=2, fontsize=9)
plt.tight_layout()
plt.savefig(REPORT_PATH / "clustering_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_23425/1010774719.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Guardar resultados

In [13]:
# Guardar modelo (bundle autocontenido: modelo + preprocesamiento + features)
with open(MODEL_PATH / "kmeans_model.pkl", "wb") as f:
    pickle.dump({"model": kmeans, "scaler": preprocess, "features": FEATURES}, f)

# Mapeo cluster -> etiqueta de negocio. Lo consume la API REST de la Entrega 04 para
# traducir la salida del K-Means sin depender del parquet completo de clientes.
with open(MODEL_PATH / "cluster_labels.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in LABELS.items()}, f, indent=2, ensure_ascii=False)

# Guardar clientes segmentados
rfm.to_parquet(OUTPUT_PATH / "clientes_segmentados.parquet", index=False)

print(f"Modelo guardado: {MODEL_PATH / 'kmeans_model.pkl'}")
print(f"Mapeo de etiquetas: {MODEL_PATH / 'cluster_labels.json'}")
print(f"Clientes segmentados: {OUTPUT_PATH / 'clientes_segmentados.parquet'}")
print(f"Shape: {rfm.shape}")

Modelo guardado: ../../data/06_models/kmeans_model.pkl
Mapeo de etiquetas: ../../data/06_models/cluster_labels.json
Clientes segmentados: ../../data/07_model_output/clientes_segmentados.parquet
Shape: (4338, 15)


## Respuesta al feedback: mejora de la separacion de clusters

La devolucion de la Entrega 03 senalo un silhouette bajo (~0.17). Comparamos el enfoque
**anterior** (9 features incluyendo Cancel_rate, log1p + estandarizacion, sin PCA) contra
el **nuevo** (8 features sin Cancel_rate, log1p + estandarizacion + PCA) para cuantificar
la mejora.

In [14]:
# Enfoque ANTERIOR (Entrega 03 original): 9 features incl. Cancel_rate, sin PCA
OLD_FEATURES = [
    "Recency",
    "Frequency",
    "Monetary",
    "Cancel_rate",
    "pct_with_color",
    "color_diversity",
    "pct_with_material",
    "avg_quantity_in_set",
    "pct_purchases_sets",
]
old_pre = Pipeline(
    [
        (
            "log",
            ColumnTransformer(
                [
                    (
                        "log1p",
                        FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
                        ["Frequency", "Monetary", "Cancel_rate"],
                    ),
                    (
                        "passthrough",
                        "passthrough",
                        [
                            f
                            for f in OLD_FEATURES
                            if f not in ["Frequency", "Monetary", "Cancel_rate"]
                        ],
                    ),
                ]
            ),
        ),
        ("scaler", StandardScaler()),
    ]
)
X_old = old_pre.fit_transform(rfm[OLD_FEATURES])

# Comparar silhouette para k=2..8 en ambos enfoques
results = []
for k in range(2, 9):
    km_old = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_old)
    km_new = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    results.append(
        {
            "k": k,
            "sil_anterior": round(silhouette_score(X_old, km_old.labels_), 3),
            "sil_nuevo": round(silhouette_score(X_scaled, km_new.labels_), 3),
        }
    )

comp = pd.DataFrame(results)
comp["mejora_%"] = ((comp["sil_nuevo"] / comp["sil_anterior"] - 1) * 100).round(0)
print(comp.to_string(index=False))

 k  sil_anterior  sil_nuevo  mejora_%
 2         0.194      0.314      62.0
 3         0.190      0.318      67.0
 4         0.174      0.314      80.0
 5         0.141      0.271      92.0
 6         0.149      0.268      80.0
 7         0.156      0.245      57.0
 8         0.165      0.247      50.0


In [15]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(comp["k"], comp["sil_anterior"], "o-", label="Anterior (9 feats, sin PCA)", color="salmon")
ax.plot(comp["k"], comp["sil_nuevo"], "o-", label="Nuevo (8 feats + PCA)", color="seagreen")
ax.axvline(best_k, ls="--", color="gray", alpha=0.6, label="k elegido = 4")
ax.set_xlabel("k (numero de clusters)")
ax.set_ylabel("Silhouette score")
ax.set_title("Separacion de clusters: antes vs despues del feedback")
ax.legend()
plt.tight_layout()
plt.savefig(REPORT_PATH / "clustering_silhouette_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_23425/4148737091.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Resumen de la mejora a k=4 (el k de negocio)
row4 = comp[comp["k"] == best_k].iloc[0]
print("=== Mejora del silhouette a k=4 ===")
print(f"  Anterior (9 feats, sin PCA): {row4['sil_anterior']:.3f}")
print(f"  Nuevo (8 feats + PCA):       {row4['sil_nuevo']:.3f}")
print(f"  Mejora relativa:             +{row4['mejora_%']:.0f}%")
print()
print("La afinidad de producto sigue presente (8 features), por lo que se conserva el")
print("segmento 'Compradores de Sets'. La mejora viene de quitar la feature ruidosa")
print("(Cancel_rate) y de-correlacionar el espacio con PCA antes de K-Means.")

=== Mejora del silhouette a k=4 ===
  Anterior (9 feats, sin PCA): 0.174
  Nuevo (8 feats + PCA):       0.314
  Mejora relativa:             +80%

La afinidad de producto sigue presente (8 features), por lo que se conserva el
segmento 'Compradores de Sets'. La mejora viene de quitar la feature ruidosa
(Cancel_rate) y de-correlacionar el espacio con PCA antes de K-Means.


## Conclusion

**Modelo:** K-Means con k=4 sobre 8 features (RFM + atributos de producto, **sin Cancel_rate**),
con log-transform, StandardScaler y **PCA (3 componentes, ~64% de varianza)**.

**Metricas (mejora respondiendo al feedback de la Entrega 03):**
- Silhouette score: **0.314** (vs 0.174 del enfoque anterior, +80%)
- Calinski-Harabasz: ~1,900 (vs ~726)
- Sin clusters degenerados (rango 13.9% - 41.2%)

**Que se cambio y por que:**
- Se **excluyo `Cancel_rate`** del clustering: 64% de ceros y outliers extremos, aportaba
  ruido. Se conserva como descriptor de negocio y como feature del modelo de churn.
- Se agrego **PCA** antes de K-Means (tecnica de reduccion de la U4) para de-correlacionar
  y compactar el espacio, mejorando la separacion sin perder la afinidad de producto.

**Segmentos identificados:**
- **VIP (41.2% clientes, 84.6% revenue):** Recientes, frecuentes, alto gasto. Core del negocio.
- **Dormidos (22.6%, 5.1% revenue):** Alta recency, baja frecuencia. Candidatos a reactivacion.
- **En Riesgo (22.2%, 5.2% revenue):** Cancel rate ~4x mayor que otros segmentos. Requieren atencion.
- **Compradores de Sets (13.9%, 5.2% revenue):** Patron de compra distintivo por sets/packs.

**Valor de negocio:**
- Confirma patron Pareto: ~41% de clientes genera ~85% del revenue.
- La afinidad de producto se mantiene en el modelo (8 features), preservando el segmento
  "Compradores de Sets" que RFM solo no detectaria.

**Proximos pasos:**
1. Modelo de prediccion de churn (notebook 08)
2. Cruzar segmentos con predicciones de churn (notebook 09)
3. Prototipo interactivo y API de scoring (Entrega 04)